# spectus Personal Notebook

Use spectus from this notebook. No HTTP server needed. The `Extractor` wraps the full pipeline in-process.

**Prereqs** (one-time, from project root):

```
uv sync --extra dev
uv run playwright install chromium
uv run alembic upgrade head
cp .env.example .env   # add your OPENAI_API_KEY
```

Then launch this notebook with `uv run jupyter lab` (or `uv run jupyter notebook`).

Optional for DataFrame display: `uv add pandas` or `pip install pandas`.

## 1. Setup

If the notebook is opened from `notebooks/`, the cell below jumps to the project root so imports resolve.

In [ ]:
import os
from pathlib import Path

here = Path.cwd()
if here.name == "notebooks":
    os.chdir(here.parent)
print("cwd:", Path.cwd())

from notebooks.spectus import Extractor

## 2. Create the extractor (one-time)

Builds browser pool + DB engine + HTTP client + all services. Reuse the same `ex` across cells — don't recreate per request.

Pass `browser=False` if you want pure static-only mode (no Chromium launch).

In [ ]:
ex = await Extractor.create(browser=True, log_level="WARNING")
print("ready")

## 3. First extraction — Hacker News front page

Pass any URL + natural-language instruction. Returns a structured `ExtractionResponse`.

In [ ]:
resp = await ex.extract(
    url="https://news.ycombinator.com/",
    instruction=(
        "Extract the top stories. For each story return title, points, author, "
        "comments_count, and story_url."
    ),
    max_records=15,
)
ex.show(resp, n=5)

## 4. Use the records

`ex.records(resp)` gives the raw list of dicts.

In [ ]:
records = ex.records(resp)
print(f"got {len(records)} records")
for r in records[:3]:
    print(r)

## 5. Display as a pandas DataFrame (optional)

Requires pandas: `uv add pandas` or `pip install pandas`. Skip this cell otherwise.

In [ ]:
try:
    df = ex.to_dataframe(resp)
    display(df)
except ImportError:
    print("pandas not installed — skip this cell or run `uv add pandas`")

## 6. Save to CSV / JSON

In [ ]:
csv_path = ex.save_csv(resp, "output/hn.csv")
json_path = ex.save_json(resp, "output/hn.json")
print("wrote", csv_path)
print("wrote", json_path)

## 7. Your own URL

Edit the cell below — paste any URL and instruction in plain English.

Tips:
- Be explicit about field names you want — they become column headers.
- Use *singular* phrasing for a single-entity page (article / contact / product detail).
- Use *plural* phrasing for list pages ("extract every product …").
- `use_browser="auto"` is the default. Use `"never"` to force static (faster, cheaper). Use `"force"` to force browser path on a known SPA.

In [ ]:
MY_URL = "https://books.toscrape.com/"
MY_INSTRUCTION = (
    "Extract every book with title, price, availability, rating, and link to the book detail page."
)

resp = await ex.extract(MY_URL, MY_INSTRUCTION, use_browser="auto", max_records=20)
ex.show(resp, n=5)

## 8. Browse saved templates

Templates are saved per `(domain, goal_signature)`. Second extraction with the same fields on the same domain reuses the template and skips the planner LLM call (3–5× faster).

In [ ]:
templates = await ex.list_templates()
print(f"saved templates: {len(templates)}\n")
for t in templates:
    print(f"  {t.domain}  pattern={t.url_pattern}  status={t.status}  "
          f"successes={t.consecutive_successes}  failures={t.consecutive_failures}  "
          f"score={t.success_score}")

## 9. Metrics snapshot

In [ ]:
import json
m = await ex.metrics()
print(json.dumps(m, indent=2, default=str))

## 10. Cleanup (when done)

Shuts down browser pool, HTTP client, DB engine. Always run before closing the notebook to avoid leaked Chromium processes.

In [ ]:
await ex.close()
print("closed")

---

## Reference

### `Extractor.extract(url, instruction, **options)`

Options:
- `use_browser`: `"auto"` (default) / `"force"` / `"never"`
- `max_records`: 1..1000 (default 100)
- `save_template`: bool (default True)

Returns `ExtractionResponse`. Useful fields:
- `resp.records` — list of dicts (or single dict for `expected_output="object"`)
- `resp.status` — `"success"` / `"partial_success"` / `"failed"`
- `resp.diagnostics` — strategy, quality_score, repair_attempts, template_used, runtime_ms, llm_tokens_*, warnings
- `resp.message` — repair hint when not good_enough

### Helper methods
- `ex.records(resp)` — flatten records to `list[dict]`
- `ex.show(resp, n=5)` — print diagnostics + first n records
- `ex.to_dataframe(resp)` — pandas DataFrame (needs pandas)
- `ex.save_csv(resp, path)` / `ex.save_json(resp, path)`
- `ex.list_templates(status=None)` — saved templates
- `ex.metrics()` — in-process counters + histograms

### Costs (gpt-4o-mini + gpt-4.1)

Per extraction with no template (cold path):
- 1 intent call (gpt-4o-mini, ~500 in / 150 out tokens)
- 1 planner call (gpt-4.1, ~4500 in / 600 out tokens)
- 0–2 repair calls (gpt-4.1, similar)

Per extraction with template hit (warm path):
- 1 intent call only (~500 in / 150 out tokens). Planner skipped entirely.

### Artifacts on disk

Every extraction writes a debug bundle to `./artifacts/{job_id}/`:
- `raw.html` / `rendered.html` (if browser used)
- `screenshot.png` (if browser used)
- `compact.json` — what the planner saw
- `intent.json` / `plan.json` / `validation.json`
- `llm/intent.1.json` / `llm/planner.1.json` / `llm/repair_1.1.json` — full LLM I/O
- `error.json` (on failure)